# Reusable Template: Binary Logistic Regression with Sigmoid

Use this notebook as a starting point for any new **binary classification** problem that can be attacked with a simple logistic model.

**Workflow**
1. Load / prepare your data (X numeric or already encoded, y ∈ {0,1})
2. Implement or import `sigmoid` and `predict_proba`
3. (Optional) Fit parameters with gradient descent or a library
4. Choose threshold according to business costs
5. Evaluate, plot, simulate sensitivity
6. Document limitations and intended use

Copy the cheat-sheet and flowchart into your project notes.


## Cheat Sheet (copy into every new project)
- \( g(z) = 1/(1+e^{-z}) \)
- \( f(x) = g(w·x + b) \)
- Decision: \( \hat y = 1\{f ≥ τ\} \)
- Always clip probabilities before taking log
- Prefer `scipy.special.expit` for production numerical stability


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit

def sigmoid(z):
    return expit(z)   # or 1/(1+np.exp(-np.clip(z,-500,500)))

def predict_proba(X, w, b):
    """X: (m,n) or (m,), w: (n,) or scalar"""
    z = X @ w + b if X.ndim > 1 else w * X + b
    return sigmoid(z)

def predict(X, w, b, threshold=0.5):
    return (predict_proba(X, w, b) >= threshold).astype(int)

def logistic_loss(X, y, w, b, eps=1e-12):
    p = np.clip(predict_proba(X, w, b), eps, 1-eps)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))


## 1. Load your data
Replace the toy data with your own CSV / array.


In [ ]:
# Example placeholder — replace with real data
# import pandas as pd
# df = pd.read_csv("your_binary_data.csv")
# X = df[["feat1","feat2"]].values
# y = df["label"].values

# Toy 1-D for illustration
X = np.array([0.,1,2,3,4,5,6,7]).reshape(-1,1)  # or keep 1-D
y = np.array([0,0,0,0,1,1,1,1])
print(X.shape, y.shape, y.mean())


## 2. Fit (simple closed-form not available; use GD or library)
Here we supply a tiny gradient-descent loop for educational purposes.


In [ ]:
def fit_logistic_gd(X, y, lr=0.1, epochs=2000, verbose=False):
    m, n = X.shape if X.ndim > 1 else (len(X), 1)
    X_ = X.reshape(m, n)
    w = np.zeros(n)
    b = 0.0
    losses = []
    for i in range(epochs):
        p = predict_proba(X_, w, b)
        # gradients
        dw = (X_.T @ (p - y)) / m
        db = np.mean(p - y)
        w -= lr * dw
        b -= lr * db
        if i % 200 == 0:
            losses.append(logistic_loss(X_, y, w, b))
            if verbose:
                print(i, losses[-1])
    return w, b, losses

w_hat, b_hat, loss_hist = fit_logistic_gd(X, y, lr=0.3, epochs=3000, verbose=True)
print("w =", w_hat, "b =", b_hat)


## 3. Evaluate & choose threshold


In [ ]:
probs = predict_proba(X, w_hat, b_hat)
print("Probs:", np.round(probs, 3))

for tau in [0.3, 0.5, 0.7]:
    acc = (predict(X, w_hat, b_hat, tau) == y).mean()
    print(f"τ={tau}  accuracy={acc:.3f}")


## 4. Visualize (1-D or first two features)


In [ ]:
if X.shape[1] == 1:
    x_line = np.linspace(X.min()-0.5, X.max()+0.5, 100).reshape(-1,1)
    fig, ax = plt.subplots(figsize=(7,4))
    ax.scatter(X[y==0], y[y==0], c='C0', label='0')
    ax.scatter(X[y==1], y[y==1], c='C3', label='1')
    ax.plot(x_line, predict_proba(x_line, w_hat, b_hat), 'k-', lw=2)
    ax.axhline(0.5, color='orange', ls='--')
    ax.legend(); ax.set_title("Fitted logistic curve")
    plt.show()
else:
    print("For >1 feature, plot decision boundary or partial dependence yourself.")


## 5. Quick sensitivity simulation


In [ ]:
# Bootstrap accuracy
rng = np.random.default_rng(0)
accs = []
for _ in range(200):
    idx = rng.integers(0, len(y), len(y))
    accs.append((predict(X[idx], w_hat, b_hat) == y[idx]).mean())
print(f"Bootstrap accuracy mean={np.mean(accs):.3f}  5-95%={np.percentile(accs,[5,95])}")


## 6. Responsible-use reminder (delete or expand for your stakeholders)
- State the population and time period the model was trained on.
- Report the chosen threshold and the resulting confusion-matrix counts.
- Note that probabilities are model assumptions, not objective truth.
- Plan for monitoring and re-training when data drift appears.
